06_text_to_sql_nordwind.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/06_text_to_sql_nordwind.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/06_text_to_sql_nordwind.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Text-to-SQL with Gemini: Querying NordWind Energy Data

## What you will learn
- Build a realistic SQLite database for a wind energy company.
- Use Gemini to convert natural language questions into SQL queries.
- See where Text-to-SQL works well and where it fails.
- Understand the promise and reality of "ask questions in English, get answers from your database."

**NLP Task**: Text-to-SQL — translating natural language into database queries.

**Scenario**: NordWind Energy operations managers want to query their maintenance
and turbine performance data without writing SQL.

Expected runtime: 10–15 minutes
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai pandas


In [ ]:
import os
import json
import time
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import random

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Build the NordWind Energy Database

A realistic SQLite database with 4 tables reflecting a wind turbine service company.


In [ ]:
random.seed(42)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# --- Table Definitions ---
cursor.executescript("""
CREATE TABLE sites (
    site_id TEXT PRIMARY KEY,
    site_name TEXT NOT NULL,
    location TEXT NOT NULL,
    turbine_count INTEGER,
    site_type TEXT CHECK(site_type IN ('onshore', 'offshore'))
);

CREATE TABLE turbines (
    turbine_id TEXT PRIMARY KEY,
    site_id TEXT REFERENCES sites(site_id),
    model TEXT NOT NULL,
    install_date DATE,
    capacity_mw REAL,
    status TEXT CHECK(status IN ('operational', 'maintenance', 'offline'))
);

CREATE TABLE technicians (
    tech_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    certification_level INTEGER CHECK(certification_level BETWEEN 1 AND 4),
    primary_site TEXT REFERENCES sites(site_id),
    specialization TEXT
);

CREATE TABLE maintenance_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    turbine_id TEXT REFERENCES turbines(turbine_id),
    tech_id TEXT REFERENCES technicians(tech_id),
    log_date DATE NOT NULL,
    category TEXT CHECK(category IN ('routine', 'corrective', 'emergency', 'predictive')),
    priority TEXT CHECK(priority IN ('low', 'medium', 'high', 'critical')),
    component TEXT,
    description TEXT,
    downtime_hours REAL DEFAULT 0,
    cost_dkk REAL DEFAULT 0
);
""")

# --- Populate Sites ---
sites = [
    ('S01', 'Ringkøbing Cluster', 'Ringkøbing, West Jutland', 12, 'onshore'),
    ('S02', 'Thyborøn Wind Farm', 'Thyborøn, North Jutland', 8, 'onshore'),
    ('S03', 'Hvide Sande Park', 'Hvide Sande, West Jutland', 15, 'onshore'),
    ('S04', 'Esbjerg Offshore', 'North Sea, 25km from Esbjerg', 20, 'offshore'),
    ('S05', 'Hanstholm Heights', 'Hanstholm, North Jutland', 6, 'onshore'),
]
cursor.executemany('INSERT INTO sites VALUES (?,?,?,?,?)', sites)

# --- Populate Turbines ---
models = ['Vestas V110', 'Vestas V136', 'Siemens SWT-3.6', 'Siemens Gamesa SG-4.5']
statuses = ['operational'] * 8 + ['maintenance'] + ['offline']
turbine_rows = []
tid = 1
for site_id, _, _, count, _ in sites:
    for i in range(count):
        turbine_id = f'NW-{tid:03d}'
        model = random.choice(models)
        install_year = random.randint(2015, 2023)
        install_date = f'{install_year}-{random.randint(1,12):02d}-{random.randint(1,28):02d}'
        capacity = round(random.uniform(2.0, 4.5), 1)
        status = random.choice(statuses)
        turbine_rows.append((turbine_id, site_id, model, install_date, capacity, status))
        tid += 1

cursor.executemany('INSERT INTO turbines VALUES (?,?,?,?,?,?)', turbine_rows)

# --- Populate Technicians ---
techs = [
    ('T01', 'Lars Henriksen', 4, 'S01', 'gearbox'),
    ('T02', 'Maja Sørensen', 3, 'S02', 'blade inspection'),
    ('T03', 'Kasper Vestergaard', 4, 'S03', 'electrical systems'),
    ('T04', 'Pernille Nielsen', 3, 'S04', 'offshore operations'),
    ('T05', 'Jonas Møller', 2, 'S01', 'general maintenance'),
    ('T06', 'Anders Christensen', 3, 'S05', 'sensors and SCADA'),
    ('T07', 'Benedikte Andersen', 2, 'S03', 'general maintenance'),
    ('T08', 'Frederik Holm', 4, 'S04', 'subsea cables'),
]
cursor.executemany('INSERT INTO technicians VALUES (?,?,?,?,?)', techs)

# --- Populate Maintenance Logs (realistic spread) ---
categories = ['routine', 'corrective', 'emergency', 'predictive']
cat_weights = [0.45, 0.25, 0.05, 0.25]
priorities = {'routine': 'low', 'corrective': 'medium', 'emergency': 'critical', 'predictive': 'medium'}
components = ['gearbox', 'blade', 'generator', 'yaw system', 'pitch system', 'transformer',
              'anemometer', 'tower bolts', 'cooling system', 'brake system']

descriptions = {
    'gearbox': ['Bearing replacement due to vibration alert', 'Oil sample analysis — elevated metal particles',
                'Gearbox alignment check after high-wind event'],
    'blade': ['Lightning strike damage on tip — surface only', 'Leading edge erosion repair',
              'Pitch bearing greasing and inspection'],
    'generator': ['Winding temperature alarm — cooling fan belt replaced', 'Generator brush replacement',
                  'Insulation resistance test — passed'],
    'yaw system': ['Yaw motor failure — replaced motor #2', 'Yaw bearing greasing',
                   'Yaw misalignment corrected — cable twist reset'],
    'pitch system': ['Pitch encoder cable replaced (rodent damage)', 'Pitch battery backup test',
                     'Pitch angle calibration'],
    'transformer': ['Transformer oil dielectric test', 'Bushing inspection — no issues',
                    'Partial discharge monitoring setup'],
    'anemometer': ['Recalibrated — was reading 12% high', 'Cup anemometer replaced (icing damage)',
                   'Ultrasonic sensor firmware update'],
    'tower bolts': ['Torque check on foundation bolts', 'Replaced corroded bolt set — section 3',
                    'Annual bolt inspection — all within spec'],
    'cooling system': ['Coolant level top-up', 'Radiator fan motor replaced',
                       'Coolant line leak repaired at nacelle junction'],
    'brake system': ['Brake pad thickness check — 4mm remaining', 'Emergency brake test — passed',
                     'Brake caliper replacement'],
}

log_rows = []
base_date = datetime(2025, 10, 1)
for log_id in range(1, 201):
    turbine = random.choice(turbine_rows)
    turbine_id = turbine[0]
    tech = random.choice(techs)
    tech_id = tech[0]
    days_offset = random.randint(0, 180)
    log_date = (base_date + timedelta(days=days_offset)).strftime('%Y-%m-%d')
    category = random.choices(categories, weights=cat_weights, k=1)[0]
    priority = priorities[category]
    if category == 'corrective':
        priority = random.choice(['medium', 'high'])
    component = random.choice(components)
    desc = random.choice(descriptions[component])
    downtime = round(random.uniform(0, 2), 1) if category == 'routine' else round(random.uniform(2, 48), 1)
    if category == 'emergency':
        downtime = round(random.uniform(12, 72), 1)
    cost = round(downtime * random.uniform(500, 3000), 0)
    log_rows.append((turbine_id, tech_id, log_date, category, priority, component, desc, downtime, cost))

cursor.executemany(
    'INSERT INTO maintenance_logs (turbine_id, tech_id, log_date, category, priority, component, description, downtime_hours, cost_dkk) VALUES (?,?,?,?,?,?,?,?,?)',
    log_rows
)
conn.commit()

# Verify
for table in ['sites', 'turbines', 'technicians', 'maintenance_logs']:
    count = cursor.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count} rows')


## Explore the Schema

This is what we'll give the LLM to understand the database.


In [ ]:
SCHEMA = """
Tables:
1. sites (site_id TEXT PK, site_name TEXT, location TEXT, turbine_count INT, site_type TEXT ['onshore','offshore'])
2. turbines (turbine_id TEXT PK, site_id TEXT FK→sites, model TEXT, install_date DATE, capacity_mw REAL, status TEXT ['operational','maintenance','offline'])
3. technicians (tech_id TEXT PK, name TEXT, certification_level INT [1-4], primary_site TEXT FK→sites, specialization TEXT)
4. maintenance_logs (log_id INT PK, turbine_id TEXT FK→turbines, tech_id TEXT FK→technicians, log_date DATE, category TEXT ['routine','corrective','emergency','predictive'], priority TEXT ['low','medium','high','critical'], component TEXT, description TEXT, downtime_hours REAL, cost_dkk REAL)

Sample data:
- Sites: Ringkøbing, Thyborøn, Hvide Sande, Esbjerg Offshore, Hanstholm
- Turbine IDs: NW-001 through NW-061
- Technicians: Lars Henriksen (gearbox specialist), Maja Sørensen (blade inspection), etc.
- Date range: October 2025 through March 2026
"""

print(SCHEMA)


## Text-to-SQL with Gemini

We ask natural language questions and let Gemini generate SQL.
Then we execute the SQL and check the results.


In [ ]:
TEXT_TO_SQL_PROMPT = """You are a SQL assistant for a wind energy maintenance database.
Given the following database schema, convert the user's natural language question into a SQLite query.

{schema}

RULES:
- Return ONLY the SQL query, no explanation.
- Use SQLite syntax.
- Do not use any tables or columns not in the schema.

Question: {question}

SQL:"""

def text_to_sql(question: str) -> dict:
    prompt = TEXT_TO_SQL_PROMPT.format(schema=SCHEMA, question=question)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        sql = resp.text.strip()
        # Clean markdown
        if sql.startswith('```'):
            sql = sql.split('\n', 1)[1] if '\n' in sql else sql[3:]
        if sql.endswith('```'):
            sql = sql.rsplit('```', 1)[0]
        sql = sql.strip()
        return {'sql': sql, 'error': None}
    except Exception as e:
        return {'sql': None, 'error': str(e)}

def execute_query(sql: str) -> dict:
    try:
        df = pd.read_sql_query(sql, conn)
        return {'result': df, 'error': None}
    except Exception as e:
        return {'result': None, 'error': str(e)}


## Test with Business Questions

These questions range from simple to complex, showing where Text-to-SQL excels and fails.


In [ ]:
QUESTIONS = [
    # Simple — should work well
    "How many turbines does NordWind operate in total?",
    "Which site has the most turbines?",
    "How many emergency maintenance events happened in the last 3 months?",

    # Medium — requires joins
    "Which technician handled the most maintenance jobs?",
    "What is the total downtime in hours for each site?",
    "Show me all gearbox-related maintenance at the Ringkøbing site.",

    # Hard — business logic + ambiguity
    "Which turbines have had the most expensive maintenance history?",
    "Are offshore turbines more expensive to maintain than onshore ones?",
    "Which components fail most often on older turbines (installed before 2018)?",

    # Tricky — ambiguous or requires domain knowledge
    "Which turbines should we worry about?",
]

results = []
for q in QUESTIONS:
    print(f'\n{"="*80}')
    print(f'Question: {q}')
    print(f'{"="*80}')

    # Generate SQL
    sql_result = text_to_sql(q)
    time.sleep(0.5)

    if sql_result['error']:
        print(f'  SQL Generation Error: {sql_result["error"]}')
        results.append({'question': q, 'sql': None, 'success': False, 'error': sql_result['error']})
        continue

    print(f'  Generated SQL: {sql_result["sql"]}')

    # Execute SQL
    exec_result = execute_query(sql_result['sql'])

    if exec_result['error']:
        print(f'  Execution Error: {exec_result["error"]}')
        results.append({'question': q, 'sql': sql_result['sql'], 'success': False, 'error': exec_result['error']})
    else:
        print(f'  Results ({len(exec_result["result"])} rows):')
        print(exec_result['result'].head(10).to_string(index=False))
        results.append({'question': q, 'sql': sql_result['sql'], 'success': True, 'error': None})


## Analyze Text-to-SQL Performance


In [ ]:
results_df = pd.DataFrame(results)
print(f'\n=== Text-to-SQL Results ===')
print(f'Success rate: {results_df["success"].mean():.0%}')
print(f'Failures:')
for _, row in results_df[~results_df['success']].iterrows():
    print(f'  Q: {row["question"]}')
    print(f'  Error: {row["error"]}')
    if row['sql']:
        print(f'  SQL attempted: {row["sql"]}')


## Discussion Questions

1. **Simple vs. complex queries**: Where did Text-to-SQL work well? Where did it fail?

2. **Ambiguity**: "Which turbines should we worry about?" requires business judgment.
   How did the model handle it? Is the SQL it generated reasonable?

3. **Joins and business logic**: Did the model correctly join tables?
   Multi-table queries are where Text-to-SQL systems typically struggle.

4. **Who gets to query data now?** If a manager can ask questions in English,
   how does that change the analyst's role?

5. **Safety**: What if a manager asks a question and gets a wrong SQL result
   that looks plausible? How would they know?

## Export


In [ ]:
results_df['student_notes'] = ''
export_path = 'text_to_sql_results.xlsx'
results_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')

conn.close()
